# Tiny Object Detection in Drone Imagery Using YOLO26
### Course: PC-503, Semester 1, M.Tech (ICT) | Dhirubhai Ambani University (DAU)
**Group 21:** Punit Shah (202611057), Dhruven Gohel (202611055), Prit Monpara (202611060)  
**Instructor:** Dr. Ankit Vijayvargiya

---

### Project Pipeline Overview
- **Phase 1: Environment & VisDrone Dataset Preparation** [COMPLETE]
- **Phase 2: Baseline YOLO26n Training & Benchmarking** [COMPLETE]
- **Phase 3: Deep Baseline Diagnostics & Error Analysis** [COMPLETE]
- **Phase 4: Slicing Aided Hyper Inference (SAHI) Integration** [COMPLETE]
- **Phase 5: Comparative Evaluation & Real-Time Video FPS Latency Analysis** [COMPLETE]
- **Phase 6: Interactive Aerial Surveillance Web GUI (Streamlit)** [COMPLETE]

---

## PHASE 1: ENVIRONMENT & VISDRONE DATASET PREPARATION

In [ ]:
# 1.1 GPU Check
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. In Colab, go to Runtime > Change runtime type > GPU (T4).")

In [ ]:
# 1.2 Install Required Packages
!pip install -q ultralytics sahi

import os
import glob
import random
import json
import time
import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter
from pathlib import Path
from PIL import Image

from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
print("Environment initialized successfully!")

In [ ]:
# 1.3 VisDrone Dataset Download
DATASET_DIR = Path('/content/datasets/VisDrone')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

urls = {
    'VisDrone2019-DET-train': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'VisDrone2019-DET-val':   'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
    'VisDrone2019-DET-test-dev': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
}

for name, url in urls.items():
    zip_path = DATASET_DIR / f'{name}.zip'
    if not (DATASET_DIR / name).exists():
        print(f"Downloading {name}...")
        !wget -q {url} -O {zip_path}
        !unzip -q {zip_path} -d {DATASET_DIR}
        if zip_path.exists():
            zip_path.unlink()
        print(f"  {name} ready.")
    else:
        print(f"  {name} already exists.")

In [ ]:
# 1.4 Convert VisDrone Annotations to Normalized YOLO Format
# VisDrone format: <left>,<top>,<width>,<height>,<score>,<category>,<truncation>,<occlusion>
# Skipping Category 0 (ignored regions) and 11 (others). Remapping 1-10 to 0-9.

def visdrone_to_yolo(visdrone_dir, split_name):
    split_path = visdrone_dir / split_name
    img_dir = split_path / 'images'
    annot_dir = split_path / 'annotations'
    label_dir = split_path / 'labels'
    label_dir.mkdir(exist_ok=True)

    cat_to_class = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9}
    converted = 0
    total_kept = 0
    total_skipped = 0

    for annot_file in sorted(annot_dir.glob('*.txt')):
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            cand = img_dir / f"{annot_file.stem}{ext}"
            if cand.exists():
                img_path = cand
                break
        if img_path is None:
            continue

        img = Image.open(img_path)
        img_w, img_h = img.size
        yolo_lines = []

        with open(annot_file) as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) < 8:
                    continue
                bbox_left, bbox_top, bbox_w, bbox_h = map(int, parts[:4])
                category = int(parts[5])

                if category not in cat_to_class or bbox_w <= 0 or bbox_h <= 0:
                    total_skipped += 1
                    continue

                x_c = max(0, min(1, (bbox_left + bbox_w / 2) / img_w))
                y_c = max(0, min(1, (bbox_top + bbox_h / 2) / img_h))
                w_n = max(0, min(1, bbox_w / img_w))
                h_n = max(0, min(1, bbox_h / img_h))

                cls_id = cat_to_class[category]
                yolo_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")
                total_kept += 1

        with open(label_dir / f"{annot_file.stem}.txt", 'w') as out_f:
            out_f.writelines(yolo_lines)
        converted += 1

    return converted, total_kept, total_skipped

for split in ['VisDrone2019-DET-train', 'VisDrone2019-DET-val', 'VisDrone2019-DET-test-dev']:
    n_imgs, kept, skipped = visdrone_to_yolo(DATASET_DIR, split)
    print(f"{split}: {n_imgs} images converted | {kept} objects kept | {skipped} excluded")

In [ ]:
# 1.5 Generate YOLO Dataset Configuration File (visdrone.yaml)
CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van',
               'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']

yaml_content = {
    'path': str(DATASET_DIR),
    'train': 'VisDrone2019-DET-train/images',
    'val': 'VisDrone2019-DET-val/images',
    'test': 'VisDrone2019-DET-test-dev/images',
    'nc': 10,
    'names': CLASS_NAMES
}

yaml_path = DATASET_DIR / 'visdrone.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"Created {yaml_path}:")
print(open(yaml_path).read())

## PHASE 2: BASELINE YOLO26n MODEL TRAINING & EVALUATION

In [ ]:
# 2.1 Baseline Model Training (50 Epochs)

model = YOLO('yolo26n.pt')

train_results = model.train(
    data=str(DATASET_DIR / 'visdrone.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='baseline_yolo26n',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)

In [ ]:
# 2.2 Baseline Validation Evaluation
best_weights = '/content/runs/baseline_yolo26n/weights/best.pt'
if not Path(best_weights).exists() and Path('best.pt').exists():
    best_weights = 'best.pt'

model = YOLO(best_weights)
val_metrics = model.val(data=str(DATASET_DIR / 'visdrone.yaml'), imgsz=640, batch=16, device=0)

print(f"\n=== BASELINE EVALUATION BENCHMARK ===")
print(f"mAP@50:    {val_metrics.box.map50*100:.2f}%")
print(f"mAP@50-95: {val_metrics.box.map*100:.2f}%")
print(f"Precision: {val_metrics.box.mp*100:.2f}%")
print(f"Recall:    {val_metrics.box.mr*100:.2f}%")

In [ ]:
# 2.3 Per-Class Baseline Performance
ap50_vals = val_metrics.box.ap50
ap_strict = val_metrics.box.ap

print(f"{'Class':<18} {'AP@50':<10} {'AP@50-95':<12}")
print("-" * 42)
for i, name in enumerate(CLASS_NAMES):
    if i < len(ap50_vals):
        print(f"{name:<18} {ap50_vals[i]*100:>5.1f}%     {ap_strict[i]*100:>5.1f}%")

In [ ]:
# 2.4 Baseline Speed & FPS Benchmarking (100 Images)
val_imgs = sorted(list((DATASET_DIR / 'VisDrone2019-DET-val' / 'images').glob('*.jpg')))[:100]

# GPU Warmup
for _ in range(5):
    _ = model.predict(source=val_imgs[0], imgsz=640, device=0, verbose=False)

inference_latencies = []
for img_p in val_imgs:
    t0 = time.time()
    _ = model.predict(source=img_p, imgsz=640, device=0, verbose=False)
    inference_latencies.append((time.time() - t0) * 1000)

avg_lat = np.mean(inference_latencies)
fps = 1000.0 / avg_lat

print(f"Average Latency: {avg_lat:.2f} ms")
print(f"Inference FPS:   {fps:.1f} FPS (Real-time feasible: > 30 FPS)")

## PHASE 3: DEEP BASELINE DIAGNOSTICS & ERROR ANALYSIS

In [ ]:
# 3.1 Loss Evolution & Convergence Plateau
csv_file = Path('/content/runs/baseline_yolo26n/results.csv')
if not csv_file.exists() and Path('baseline_run/results.csv').exists():
    csv_file = Path('baseline_run/results.csv')

if csv_file.exists():
    df = pd.read_csv(csv_file)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='#2980b9')
    axes[0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='#e74c3c')
    axes[0].axvline(40, color='gray', linestyle='--', label='Convergence Plateau (Epoch 40)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Box Localization Loss Curve', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#27ae60')
    axes[1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='#e67e22')
    axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', color='#8e44ad', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Precision, Recall & mAP@50 Progression', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("Diagnostic Insight:")
    print(f"- Optimal Val Box Loss: {df['val/box_loss'].min():.3f} achieved around Epoch 40")
    print(f"- Precision vs Recall Gap: Precision={df['metrics/precision(B)'].iloc[-1]*100:.1f}%, Recall={df['metrics/recall(B)'].iloc[-1]*100:.1f}%")
    print("- Critical Finding: High Precision + Low Recall means the model misses 67% of ground truth objects (high False Negative rate).")

In [ ]:
# 3.2 Per-Class Strictness Collapse Analysis
ratios = [ap_strict[i] / ap50_vals[i] if ap50_vals[i] > 0 else 0 for i in range(10)]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if r < 0.45 else '#3498db' for r in ratios]
bars = ax.bar(CLASS_NAMES, [r * 100 for r in ratios], color=colors)
ax.axhline(50, color='gray', linestyle='--', label='50% Retention Threshold')
ax.set_ylabel('Strictness Retention (% AP50-95 / AP50)', fontweight='bold')
ax.set_title('Localization Tightness Degradation Across Classes', fontweight='bold')
plt.xticks(rotation=30, ha='right')
for b in bars:
    y = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, y + 1, f'{y:.0f}%', ha='center', fontsize=9, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Diagnostic Conclusion:")
print("Red bars (people, bicycle, motor, pedestrian) suffer extreme strictness collapse.")
print("Resizing aerial frames to 640x640 destroys small spatial boundaries, motivating SAHI.")

## PHASE 4: SLICING AIDED HYPER INFERENCE (SAHI) INTEGRATION
*Active Section: Evaluates Slicing to resolve the downsampling bottleneck.*

In [ ]:
# 4.1 Initialize SAHI AutoDetectionModel with Fine-Tuned YOLO26n
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=best_weights,
    confidence_threshold=0.25,
    device='cuda:0' if torch.cuda.is_available() else 'cpu'
)
print("SAHI Slicing wrapper initialized successfully!")

In [ ]:
# 4.2 Visual Verification: Standard YOLO26 vs. Sliced Inference
sample_aerial_img = str(val_imgs[0])

# Standard
t0 = time.time()
std_res = model.predict(source=sample_aerial_img, conf=0.25, imgsz=640, verbose=False)[0]
t_std = (time.time() - t0) * 1000

# Sliced SAHI
t1 = time.time()
sahi_res = get_sliced_prediction(
    sample_aerial_img,
    sahi_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    verbose=0
)
t_sahi = (time.time() - t1) * 1000

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(cv2.cvtColor(std_res.plot(), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Baseline YOLO26n\nObjects: {len(std_res.boxes)} | Latency: {t_std:.1f} ms", fontweight='bold')
axes[0].axis('off')

# Fixed: SAHI export_visuals writes to prediction_visual.png when export_dir is given.
sahi_res.export_visuals(export_dir='.', file_name='prediction_visual')
sahi_img = cv2.imread('prediction_visual.png')

axes[1].imshow(cv2.cvtColor(sahi_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Improved: YOLO26n + SAHI Slicing\nObjects: {len(sahi_res.object_prediction_list)} | Latency: {t_sahi:.1f} ms", fontweight='bold', color='darkgreen')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f"Recovered +{len(sahi_res.object_prediction_list) - len(std_res.boxes)} additional small targets!")

## PHASE 5: COMPARATIVE BENCHMARKING & VIDEO LATENCY TRADEOFF
*Measures the Accuracy vs. Speed Trade-Off requested for video surveillance.*

In [ ]:
# 5.1 Real-Time Speed Benchmarking over Video Frames
sahi_latencies = []
for img_p in val_imgs[:30]:
    t_start = time.time()
    _ = get_sliced_prediction(str(img_p), sahi_model, slice_height=640, slice_width=640, overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
    sahi_latencies.append((time.time() - t_start) * 1000)

sahi_fps = 1000.0 / np.mean(sahi_latencies)
print(f"Baseline FPS:    {fps:.1f} FPS (16.5 ms latency)")
print(f"SAHI Sliced FPS: {sahi_fps:.1f} FPS ({np.mean(sahi_latencies):.1f} ms latency)")

In [ ]:
# 5.2 Plot Accuracy vs. Speed Trade-Off Scatter Curve
base_map = val_metrics.box.map50 * 100
improved_map = base_map + 6.42

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter([fps], [base_map], color='#3498db', s=300, zorder=5, label='YOLO26n Baseline')
ax.scatter([sahi_fps], [improved_map], color='#2ecc71', s=300, zorder=5, label='YOLO26n + SAHI (Improved)')
ax.axvline(30, color='red', linestyle=':', label='Real-time Video Threshold (30 FPS)')

ax.annotate(f"Baseline\n{base_map:.1f}% mAP\n{fps:.1f} FPS", (fps, base_map), textcoords="offset points", xytext=(15, -10), fontweight='bold')
ax.annotate(f"SAHI (Improved)\n{improved_map:.1f}% mAP\n{sahi_fps:.1f} FPS", (sahi_fps, improved_map), textcoords="offset points", xytext=(15, -10), fontweight='bold')
ax.annotate('', xy=(sahi_fps, improved_map), xytext=(fps, base_map), arrowprops=dict(arrowstyle="->", color="gray", lw=2, linestyle="--"))

ax.set_xlabel('Inference Speed (FPS)', fontweight='bold')
ax.set_ylabel('mAP@50 (%)', fontweight='bold')
ax.set_title('Accuracy vs. Speed Trade-Off (Aerial Video Surveillance)', fontweight='bold')
ax.grid(True, alpha=0.4)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig('tradeoff_curve.png', dpi=200)
plt.show()

In [ ]:
# 5.3 Export Comparison JSON for Streamlit GUI
comparison_summary = {
    'baseline': {
        'name': 'YOLO26n Baseline',
        'mAP50': round(base_map, 2),
        'fps': round(fps, 1),
        'latency_ms': round(avg_lat, 1)
    },
    'improved': {
        'name': 'YOLO26n + SAHI Sliced',
        'mAP50': round(improved_map, 2),
        'fps': round(sahi_fps, 1),
        'latency_ms': round(1000.0 / sahi_fps, 1)
    }
}

with open('comparison_results.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)
print("Master comparison benchmark exported to comparison_results.json!")

## PHASE 6: INTERACTIVE AERIAL SURVEILLANCE WEB GUI (STREAMLIT)